# Aether Stage 2 — Phase 7B live microphone check on RTX 3060

This notebook runs the complete live path from a recorded WAV file:

`audio -> Mimi q0 -> AetherSpeech -> R1 Connector -> Qwen3-4B -> text`

It downloads the private Stage 1 and Stage 2 checkpoints, loads Qwen3-4B in 4-bit for a 12 GB RTX 3060, transcribes the recording, and then performs an experimental direct-answer prompt on the same speech states.

Record the WAV first from the repository root:

```bash
uv run --with sounddevice python scripts/record_microphone.py --output recordings/question.wav
```


In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-e",
    ".",
    "bitsandbytes",
    "accelerate",
])
print("SETUP: dependencies ready", flush=True)


In [ ]:
import gc
import getpass
import hashlib
import json
import logging
import os
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch
import torchaudio

from aether_v3.config import load_config

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)

AUDIO_PATH = Path("recordings/question.wav")
CONFIG_PATH = Path("configs/stage2_r1_full.yaml")
STAGE1_REPO = "manifestro/aetherASR-EN-v0.1"
STAGE1_FILE = "last.pt"
STAGE2_REPO = "manifestro/aether"
STAGE2_FILE = "stage2/best_wer.pt"
OUTPUT_PATH = Path("recordings/question-result.json")

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face read token: ")

assert AUDIO_PATH.exists(), f"Record or copy the WAV first: {AUDIO_PATH}"
assert CONFIG_PATH.exists(), f"Run the notebook from the repository root: {CONFIG_PATH}"
assert torch.cuda.is_available(), "CUDA GPU is required"

cfg = load_config(CONFIG_PATH)
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
print("VRAM:", torch.cuda.get_device_properties(0).total_memory / 2**30, "GB", flush=True)
print("AUDIO:", AUDIO_PATH.resolve(), flush=True)


In [ ]:
from huggingface_hub import hf_hub_download

stage1_path = hf_hub_download(
    repo_id=STAGE1_REPO,
    filename=STAGE1_FILE,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
stage2_path = hf_hub_download(
    repo_id=STAGE2_REPO,
    filename=STAGE2_FILE,
    token=os.environ["HF_TOKEN"],
)
print("STAGE 1:", stage1_path, flush=True)
print("STAGE 2:", stage2_path, flush=True)


In [ ]:
waveform, sample_rate = sf.read(AUDIO_PATH, dtype="float32", always_2d=True)
waveform = waveform.mean(axis=1)
assert waveform.size > 0, "The WAV file is empty"
assert np.isfinite(waveform).all(), "The WAV contains NaN or Inf"

if sample_rate != cfg.mimi.sampling_rate:
    waveform = torchaudio.functional.resample(
        torch.from_numpy(waveform),
        sample_rate,
        cfg.mimi.sampling_rate,
    ).numpy()
    sample_rate = cfg.mimi.sampling_rate

duration = len(waveform) / sample_rate
peak = float(np.abs(waveform).max(initial=0.0))
assert duration >= 0.25, f"Recording is too short: {duration:.2f}s"
assert peak > 1e-4, f"Recording appears silent: peak={peak}"
print(
    "AUDIO VERIFIED:",
    {"seconds": duration, "sample_rate": sample_rate, "peak": peak},
    flush=True,
)


In [ ]:
from aether_v3.models.mimi_wrapper import FrozenMimi

print("MIMI: loading and encoding q0 semantic tokens", flush=True)
mimi_started = time.time()
mimi = FrozenMimi(
    cfg.mimi.pretrained_id,
    cfg.mimi.num_quantizers,
    device=device,
)
semantic_codes = mimi.encode_semantic([waveform])[0]
assert semantic_codes.ndim == 1 and len(semantic_codes) > 0
assert int(semantic_codes.min()) >= 0
assert int(semantic_codes.max()) < cfg.aether_speech.semantic_vocab_size
print(
    "MIMI COMPLETE:",
    {"frames": len(semantic_codes), "seconds": time.time() - mimi_started},
    flush=True,
)

del mimi
gc.collect()
torch.cuda.empty_cache()
print("MIMI RELEASED; allocated GB:", torch.cuda.memory_allocated() / 2**30, flush=True)


In [ ]:
import dataclasses

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.training.stage2_utils import load_stage1_encoder
from aether_v3.training.train_stage2 import load_stage2_trainable_weights

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("QWEN: loading Qwen3-4B in 4-bit", flush=True)
tokenizer = AutoTokenizer.from_pretrained(
    cfg.llm.model_id,
    revision=cfg.llm.revision,
    token=os.environ["HF_TOKEN"],
)
llm = AutoModelForCausalLM.from_pretrained(
    cfg.llm.model_id,
    revision=cfg.llm.revision,
    token=os.environ["HF_TOKEN"],
    quantization_config=quantization,
    device_map={"": 0},
    dtype=torch.float16,
    attn_implementation="sdpa",
)

inference_llm_cfg = dataclasses.replace(
    cfg.llm,
    dtype="float16",
    gradient_checkpointing=False,
)
model = AetherSpeechLLM(
    cfg.aether_speech,
    cfg.connector,
    inference_llm_cfg,
    speech_frozen=True,
    llm=llm,
)

stage1_checkpoint = load_stage1_encoder(stage1_path, model.encoder)
model.encoder.to(device)
model.connector.to(device=device, dtype=torch.float16)
stage2_checkpoint = load_stage2_trainable_weights(model, stage2_path, device)
model.eval()

assert model.connector.bridge.output_scale.dtype == torch.float32
print(
    "MODEL READY:",
    {
        "stage1_step": stage1_checkpoint.get("step"),
        "stage2_step": stage2_checkpoint.get("step"),
        "output_scale": float(model.connector.bridge.output_scale.detach()),
        "allocated_gb": torch.cuda.memory_allocated() / 2**30,
    },
    flush=True,
)


In [ ]:
codes = torch.from_numpy(semantic_codes).to(device=device, dtype=torch.long).unsqueeze(0)
speech_mask = torch.ones_like(codes, dtype=torch.bool)

with torch.inference_mode():
    speech_states = model.encoder(codes, speech_mask)

assert speech_states.shape == (1, len(semantic_codes), cfg.aether_speech.hidden_size)
print("AETHERSPEECH:", tuple(speech_states.shape), flush=True)


In [ ]:
def generate_for_prompt(prompt, max_new_tokens):
    prefix = tokenizer(prompt, add_special_tokens=False, return_tensors="pt")
    prefix_ids = prefix["input_ids"].to(device)
    prefix_mask = prefix["attention_mask"].to(device=device, dtype=torch.bool)
    batch = {
        "speech_states": speech_states,
        "speech_mask": speech_mask,
        "prefix_ids": prefix_ids,
        "prefix_mask": prefix_mask,
    }
    started = time.time()
    token_ids = model.generate_cached(
        batch,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=max_new_tokens,
        use_kv_cache=True,
    )[0]
    return tokenizer.decode(token_ids, skip_special_tokens=True).strip(), time.time() - started

transcription_prompt = (
    "Transcribe the following speech exactly. Output only the transcript:\n"
)
answer_prompt = (
    "Answer the spoken question directly. Give only a short answer:\n"
)

transcript, transcription_seconds = generate_for_prompt(transcription_prompt, 256)
answer, answer_seconds = generate_for_prompt(answer_prompt, 96)

print("\nTRANSCRIPT:\n", transcript, sep="", flush=True)
print("\nDIRECT ANSWER (EXPERIMENTAL):\n", answer, sep="", flush=True)
print(
    "\nTIMING:",
    {
        "transcription_seconds": transcription_seconds,
        "answer_seconds": answer_seconds,
        "peak_allocated_gb": torch.cuda.max_memory_allocated() / 2**30,
    },
    flush=True,
)


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

result = {
    "audio_path": str(AUDIO_PATH.resolve()),
    "audio_sha256": sha256_file(AUDIO_PATH),
    "audio_seconds": duration,
    "sample_rate": sample_rate,
    "semantic_frames": len(semantic_codes),
    "stage1_checkpoint": stage1_path,
    "stage1_step": stage1_checkpoint.get("step"),
    "stage2_checkpoint": stage2_path,
    "stage2_step": stage2_checkpoint.get("step"),
    "transcript": transcript,
    "direct_answer_experimental": answer,
    "transcription_seconds": transcription_seconds,
    "answer_seconds": answer_seconds,
    "gpu": torch.cuda.get_device_name(0),
    "peak_allocated_gb": torch.cuda.max_memory_allocated() / 2**30,
}
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.write_text(json.dumps(result, indent=2, ensure_ascii=False))
print("RESULT SAVED:", OUTPUT_PATH.resolve(), flush=True)
